In [6]:
import re
from bs4 import BeautifulSoup

with open("raw_html/Costco_yogurt.html", "r", encoding='utf-8') as file:
    costco_html = file.read()

soup = BeautifulSoup(costco_html, "html.parser")

In [ ]:
product_list_main = soup.find('div', attrs={'role': 'region', 'aria-label': re.compile(r'results for', re.IGNORECASE)})
product_cards_main = product_list_main.find_all('div', recursive=False)
div_num = 1
for card in product_cards_main:
    if card.find('div', attrs={'data-testid': re.compile(r'display-ad', re.IGNORECASE)}):
        div_num += 1
        continue
    product_cards = card.find_all('div', attrs = {'aria-label': re.compile(r'product', re.IGNORECASE)})
    if product_cards:
        for product in product_cards:
            title_div = product.find('h3')
            if title_div:
                title_text = title_div.get_text(strip=True)

            price_span = product.find('span', string=re.compile('current price', re.IGNORECASE))
            if price_span: 
                price_text = price_span.get_text(strip=True)
                match = re.search(r'\$?([\d\.]+)', price_text)
                price = float(match.group(1).replace(',', '')) if match else None

            img_tag = product.find('img', attrs={'data-testid': re.compile(r'item-card-image', re.IGNORECASE)})
            if img_tag:
                img_srcset = img_tag['srcset'] if 'srcset' in img_tag.attrs else None
                if img_srcset:
                    entries = re.split(r',\s+', img_srcset.strip())
                    img_url = entries[-1].split(' ')[0]

            product_link_anchor = card.find('a', href=True, role='button')
            if product_link_anchor:
                product_link = f'https://sameday.costco.com{product_link_anchor["href"]}'

            print(f'product: {title_text}\nprice: {price}\nimage: {img_url}\nlink: {product_link}\n')

    div_num += 1

product: Stonyfield Organic Yogurt Pouch, Variety Pack, 3.5 oz, 16-count
price: 19.28
image_srcset: https://www.instacart.com/image-server/197x197/filters:fill(FFFFFF,true):format(jpg)/d2lnr5mha7bycj.cloudfront.net/product-image/file/large_a69928f5-5233-4b50-9f67-ec757ff54abf.jpg, https://www.instacart.com/image-server/296x296/filters:fill(FFFFFF,true):format(jpg)/d2lnr5mha7bycj.cloudfront.net/product-image/file/large_a69928f5-5233-4b50-9f67-ec757ff54abf.jpg 1.5x, https://www.instacart.com/image-server/394x394/filters:fill(FFFFFF,true):format(jpg)/d2lnr5mha7bycj.cloudfront.net/product-image/file/large_a69928f5-5233-4b50-9f67-ec757ff54abf.jpg 2x, https://www.instacart.com/image-server/591x591/filters:fill(FFFFFF,true):format(jpg)/d2lnr5mha7bycj.cloudfront.net/product-image/file/large_a69928f5-5233-4b50-9f67-ec757ff54abf.jpg 3x, https://www.instacart.com/image-server/788x788/filters:fill(FFFFFF,true):format(jpg)/d2lnr5mha7bycj.cloudfront.net/product-image/file/large_a69928f5-5233-4b50-9f